# 01 — Contract Specifications and Rolls

**Purpose:** Pin down verified contract arithmetic (multipliers, ticks) and establish the roll-audit methodology that decides how continuous vs mapped series are used (mandate §8.2–§8.4).

**Research questions:**
1. Are the recorded contract specs internally consistent and correctly loaded?
2. Where are the roll events, and do adjusted continuous series show artificial return jumps around them?
3. What post-roll warm-up and pre-roll exclusion windows should signals use?
4. Should open trades be closed before rolls?

**Data used:** `config/instruments.yaml`, `data/metadata/contract_specifications.csv` (specs — populated); mapped+continuous futures series (roll audit — **blocked**, L-001).

**Assumptions under test:** A-003 (spec correctness), A-004 (continuous-for-indicators / mapped-for-execution split, D-004).


In [1]:
import sys
sys.path.insert(0, "../src")
import numpy as np
import pandas as pd
import yaml

RESEARCH_CONFIG = yaml.safe_load(open("../config/research_config.yaml"))
SEED = RESEARCH_CONFIG["meta"]["random_seed"]
np.random.seed(SEED)
print(f"config loaded | global seed = {SEED}")


config loaded | global seed = 20260801


In [2]:
from spread_research.contract_metadata import load_contract_specs
specs = load_contract_specs("../config/instruments.yaml")
meta = pd.read_csv("../data/metadata/contract_specifications.csv")
meta[["symbol", "name", "multiplier_or_point_value", "tick_size", "tick_value_usd",
      "expiry_cycle", "settlement", "verification_status"]]

,symbol,name,multiplier_or_point_value,tick_size,tick_value_usd,expiry_cycle,settlement,verification_status
0,MES,Micro E-mini S&P 500,5.0,0.250000,1.2500,"H,M,U,Z",cash,secondary_sources_agree_cme_unreachable
1,MNQ,Micro E-mini Nasdaq-100,2.0,0.250000,0.5000,"H,M,U,Z",cash,secondary_sources_agree_cme_unreachable
2,M2K,Micro E-mini Russell 2000,5.0,0.100000,0.5000,"H,M,U,Z",cash,secondary_sources_agree_cme_unreachable
3,MYM,Micro E-mini Dow,0.5,1.000000,0.5000,"H,M,U,Z",cash,secondary_sources_agree_cme_unreachable
4,ZT,2-Year US Treasury Note,2000.0,0.003906,7.8125,"H,M,U,Z",physical_delivery,secondary_sources_agree_cme_unreachable
5,ZF,5-Year US Treasury Note,1000.0,0.007812,7.8125,"H,M,U,Z",physical_delivery,secondary_sources_agree_cme_unreachable_face_v...
6,ZN,10-Year US Treasury Note,1000.0,0.015625,15.6250,"H,M,U,Z",physical_delivery,secondary_sources_agree_cme_unreachable
7,ZB,30-Year US Treasury Bond,1000.0,0.031250,31.2500,"H,M,U,Z",physical_delivery,secondary_sources_agree_cme_unreachable


In [3]:
# Tick arithmetic spot checks (these numbers feed every cost calculation)
checks = []
for sym, s in specs.items():
    checks.append({
        "symbol": sym,
        "tick_value_recomputed": s.tick_size * s.multiplier,
        "tick_value_recorded": s.tick_value,
        "example_notional": s.notional({"MES": 5000, "MNQ": 18000, "M2K": 2200,
                                        "MYM": 40000, "ZT": 103, "ZF": 108,
                                        "ZN": 112, "ZB": 120}[sym]),
    })
pd.DataFrame(checks)

,symbol,tick_value_recomputed,tick_value_recorded,example_notional
0,MES,1.2500,1.2500,25000.0
1,MNQ,0.5000,0.5000,36000.0
2,M2K,0.5000,0.5000,11000.0
3,MYM,0.5000,0.5000,20000.0
4,ZT,7.8125,7.8125,206000.0
5,ZF,7.8125,7.8125,108000.0
6,ZN,15.6250,15.6250,112000.0
7,ZB,31.2500,31.2500,120000.0


## Roll-audit methodology (executes when data is available)

For each instrument (mandate §8.4):
1. `contract_mapping.detect_mapping_changes` on the mapped-contract column → roll-event table.
2. `roll_adjustment.roll_gap_report` — raw price gap vs adjusted-series return at each roll; `artifact_flag` marks adjusted returns >10× local MAD (a normalization artifact).
3. `visualization.plot_roll_events` — visual inspection panels of raw vs adjusted around each roll.
4. Z-score behavior across rolls: recompute notebook-05 z-scores with and without `contract_mapping.roll_exclusion_mask`; quantify the excess |z|>2 event rate attributable to rolls.
5. Decide (and log in `research_decisions.md`): pre-roll entry exclusion days, post-roll warm-up bars, whether `flat_before_roll` stays true, and whether statistics reset or carry across mapping changes.

Output: `reports/validation/01_data_and_roll_validation.md`.


In [4]:
from pathlib import Path

DATA_DIR = Path("../data/processed")
DATA_AVAILABLE = any(DATA_DIR.glob("*_minute.*")) if DATA_DIR.exists() else False
if not DATA_AVAILABLE:
    print("BLOCKED-ON-DATA: no futures market data in this environment (see "
          "reports/00_repository_audit.md, issue L-001).\n"
          "Run this notebook inside QuantConnect Research, or drop licensed data\n"
          "into data/processed/ in the canonical schema (src/spread_research/data_loader.py).")


BLOCKED-ON-DATA: no futures market data in this environment (see reports/00_repository_audit.md, issue L-001).
Run this notebook inside QuantConnect Research, or drop licensed data
into data/processed/ in the canonical schema (src/spread_research/data_loader.py).


In [5]:
if DATA_AVAILABLE:
    from spread_research.data_loader import load_local
    from spread_research.contract_mapping import detect_mapping_changes, roll_exclusion_mask
    from spread_research.roll_adjustment import roll_gap_report
    from spread_research.visualization import plot_roll_events
    for sym in specs:
        df = load_local(sym, "minute", DATA_DIR)   # requires mapped_contract column
        if "mapped_contract" not in df.columns:
            print(sym, ": no mapped_contract column — cannot audit rolls"); continue
        events = detect_mapping_changes(df["mapped_contract"])
        print(sym, f"{len(events)} roll events")
        # raw vs adjusted comparison requires both series; see data_loader docstring
else:
    print("Roll audit BLOCKED-ON-DATA (L-001)")

Roll audit BLOCKED-ON-DATA (L-001)


## Results

**BLOCKED-ON-DATA** — this section intentionally contains no results. No synthetic or fabricated market findings are presented as evidence (CLAUDE.md gate 3). It will be populated when the notebook runs against real data. Spec-verification cells above DID run; only the roll audit is blocked.

## Limitations

- Specs verified against multiple secondary sources only; CME primary pages unreachable from this container (A-003). One source's ZF face-value error was caught and corrected — a concrete reminder that single-source specs are unsafe.
- Treasury last-trading-day rules recorded as "verify": exact delivery-cycle handling must be confirmed against CME rules before the roll audit is finalized.

## Decision

Specs adopted for research arithmetic with re-verification flagged. Roll-window parameters remain candidates (`research_config.yaml: roll_exclusion_days, post_roll_warmup_bars`) until the audit runs.


## What this means for the algorithm

Tick values and multipliers here are the foundation of every cost and PnL figure downstream. The continuous/mapped split (D-004) remains a hypothesis until the roll audit confirms adjusted series are artifact-free — if they are not, indicator construction must switch to per-contract stitching with explicit warm-ups.